In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import xarray as xr

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from neuralop.models.local_no import LocalNO
from neuralop.models import UNO
from neuralop.layers.spectral_convolution import SpectralConv

torch.serialization.add_safe_globals([torch.nn.functional.gelu])

In [ ]:
class ContinuousBasisConv3d(nn.Module):
    def __init__(self, channels, kernel_size=5, num_bases=6, h=1.0, is_differential=False):
        super().__init__()
        self.channels = channels
        self.kernel_size = kernel_size
        self.num_bases = num_bases
        self.h = h
        self.is_differential = is_differential

        pad = kernel_size // 2
        grid_t, grid_y, grid_x = torch.meshgrid(
            torch.arange(-pad, pad + 1),
            torch.arange(-pad, pad + 1),
            torch.arange(-pad, pad + 1),
            indexing='ij'
        )
        self.register_buffer('grid_t', grid_t.float())
        self.register_buffer('grid_y', grid_y.float())
        self.register_buffer('grid_x', grid_x.float())

        self.time_scale = nn.Parameter(torch.tensor(1.0))
        self.basis_params = nn.Parameter(torch.ones(num_bases) * 0.5)
        
        # Инициализация Кайминга
        fan_in = channels * (kernel_size ** 3)
        std = (2.0 / fan_in) ** 0.5
        self.basis_weights = nn.Parameter(torch.randn(channels, channels, num_bases) * std)
        self.bias = nn.Parameter(torch.zeros(channels))

    def generate_kernel(self):
        dx = self.grid_x * self.h
        dy = self.grid_y * self.h
        dt = self.grid_t * self.time_scale
        r_sq = dt**2 + dx**2 + dy**2
        r = torch.sqrt(r_sq + 1e-8)
    
        params = torch.clamp(torch.abs(self.basis_params), min=0.1)
        p = params.view(-1, 1, 1, 1)
    
        half = self.num_bases // 2

        gauss = torch.exp(-(r_sq.unsqueeze(0) / (p[:half]**2)))
        hats = F.relu(1.0 - (r.unsqueeze(0) / p[half:]))
        
        bases_tensor = torch.cat([gauss, hats], dim=0)
        kernel = torch.einsum('oib, btyx -> oityx', self.basis_weights, bases_tensor)
        kernel = kernel / np.sqrt(self.num_bases)
        
        if self.is_differential:
            kernel_mean = kernel.mean(dim=(2, 3, 4), keepdim=True)
            kernel = kernel - kernel_mean
            
        return kernel

    def forward(self, x):
        weight = self.generate_kernel()
        return F.conv3d(x, weight, bias=self.bias, padding=self.kernel_size // 2)

In [ ]:
class TrueLocalNOBlock3d(nn.Module):
    def __init__(self, channels, modes):
        super().__init__()
        self.fno_conv = SpectralConv(in_channels=channels, out_channels=channels, n_modes=modes)
        self.local_integral = ContinuousBasisConv3d(channels, kernel_size=5, num_bases=6, is_differential=False)
        self.local_differential = ContinuousBasisConv3d(channels, kernel_size=5, num_bases=6, is_differential=True)
        self.skip = nn.Conv3d(channels, channels, kernel_size=1)
        
        self.norm = nn.InstanceNorm3d(channels)
        self.activation = nn.GELU()
        self.fusion_weights = nn.Parameter(torch.ones(4)) 

    def forward(self, x, mask=None):
        identity = x 
        w = torch.softmax(self.fusion_weights, dim=0)
        
        out_fno = self.fno_conv(x)
        out_int = self.local_integral(x)
        out_diff = self.local_differential(x)
        out_skip = self.skip(x)
        
        out = (w[0] * out_fno + w[1] * out_int + w[2] * out_diff + w[3] * out_skip)
        
        out = self.norm(identity + out)
        if mask is not None:
            out = out * mask
            
        return self.activation(out)


class CustomLocalNO3d(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, hidden_channels=32, n_layers=4, modes=(10, 20, 20)):
        super().__init__()
        self.lift = nn.Conv3d(in_channels, hidden_channels, kernel_size=1)
        self.blocks = nn.ModuleList(
            [TrueLocalNOBlock3d(channels=hidden_channels, modes=modes) for _ in range(n_layers)]
        )
        
        self.project = nn.Sequential(
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=1),
            nn.GELU(),
            nn.Conv3d(hidden_channels, out_channels, kernel_size=1)
        )

    def forward(self, x, mask=None):
        x = self.lift(x)
        if mask is not None:
            x = x * mask
        
        for block in self.blocks:
            x = block(x, mask=mask)
        x = self.project(x)
        return x

In [ ]:
CURRENT_DIRECTORY = f'horizontal_correct/'
DATA_PATH = f'data'
ASSETS_PATH = f'assets'
MODELS_PATH = f'models'

filepaths =[f'{DATA_PATH}/source_{i}.hdf5' for i in range(1, 4)]

dim_map = {
    'phony_dim_0' : 'time',
    'phony_dim_1' : 'sample',
    'phony_dim_2' : 'channel',
    'phony_dim_3' : 'Y',
    'phony_dim_4' : 'X'
}

In [ ]:
def preprocess(path):
    print(f'Открытие датасета : {path}')

    ds = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort').rename(dim_map)

    pressure = ds.dataset.isel(channel=0)
    source = ds.source
    
    return xr.Dataset({'pressure': pressure, 'source':source}, attrs=ds.attrs)

print("Загрузка данных...")
processed_datasets = [preprocess(f) for f in filepaths]
final_ds = xr.concat(processed_datasets, dim='sample').squeeze('phony_dim_5')

In [ ]:
raw_p = final_ds.pressure.values
mask_2d = ~np.isnan(raw_p[0, 0])

p_min, p_max = np.nanmin(raw_p), np.nanmax(raw_p)
norm_p = np.zeros_like(raw_p, dtype='float32')
norm_p[:, :, mask_2d] = (raw_p[:, :, mask_2d] - p_min) / (p_max - p_min)

s_raw = final_ds.source.values.copy()
s_raw[..., ~mask_2d] = 0 
s_abs_max = np.max(np.abs(s_raw))
norm_s = (s_raw / s_abs_max).astype('float32') if s_abs_max != 0 else s_raw

In [ ]:
P_tensor = torch.from_numpy(norm_p).permute(1, 0, 2, 3).float()
S_tensor = torch.from_numpy(norm_s).float()
STATIC_MASK = torch.from_numpy(mask_2d.astype('float32'))

X_final = torch.stack([P_tensor, S_tensor], dim=2)

In [ ]:
def get_indices(size, train_ratio=0.8, val_ratio=0.1):
    np.random.seed(2318)

    indices = np.arange(size)
    np.random.shuffle(indices)

    train_end = int(size * train_ratio)
    val_end = train_end + int(size * val_ratio)

    return indices[:train_end], indices[train_end:val_end], indices[val_end:]

In [ ]:
class SpatiotemporalDataset(Dataset):

    def __init__(self, data, indices):
        self.data = data[indices] 
        self.N_samples = self.data.shape[0]
        self.T_total = self.data.shape[1]
        self.T_future = self.T_total - 1

    def __len__(self):
        return self.N_samples

    def __getitem__(self, idx):
        p_init = self.data[idx, 0, 0] 
        p_init_cube = p_init.unsqueeze(0).repeat(self.T_future, 1, 1)

        s_future_cube = self.data[idx, 1:, 1]

        x = torch.stack([p_init_cube, s_future_cube], dim=0)
        y = self.data[idx, 1:, 0].unsqueeze(0)

        return x, y

In [ ]:
def train_model(modes, hidden_channels, n_layers, train_loader, val_loader, device, static_mask, epochs=150):
    
    torch.backends.cudnn.benchmark = True 
    
    model = CustomLocalNO3d(
        in_channels=2, 
        out_channels=1, 
        hidden_channels=hidden_channels, 
        n_layers=n_layers, 
        modes=modes
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3) 

    mask = static_mask.to(device).float()
    mask_3d = mask.unsqueeze(0).unsqueeze(0).unsqueeze(0) 

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    best_val_l2 = float('inf')
    best_model_path = "localno_rbf_ver4.pth"
    patience_counter = 0

    print(f"=== START TRAINING | Device: {device} ===")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for batch in train_loader:
            x_in, y_true = [b.to(device) for b in batch]
            optimizer.zero_grad()
            
            pred = model(x_in, mask=mask_3d) 
            loss = criterion(pred * mask_3d, y_true * mask_3d)
            
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_train_loss += loss.item() * x_in.size(0)
            
        epoch_train_loss = total_train_loss / len(train_loader.dataset)

        model.eval()
        val_l2_errors =[]
        with torch.no_grad():
            for batch_v in val_loader:
                x_v, y_v =[b.to(device) for b in batch_v]
                
                pred_v = model(x_v, mask=mask_3d) 
                
                pred_v = pred_v * mask_3d
                true_v = y_v * mask_3d
                
                preds_flat = pred_v.reshape(x_v.size(0), -1)
                trues_flat = true_v.reshape(x_v.size(0), -1)
                
                diff_norm = torch.norm(preds_flat - trues_flat, p=2, dim=1).float()
                true_norm = torch.norm(trues_flat, p=2, dim=1).float()
                rel_l2 = (diff_norm / torch.clamp(true_norm, min=1e-8)) * 100
                val_l2_errors.extend(rel_l2.cpu().numpy())

        epoch_mean = np.mean(val_l2_errors)
        scheduler.step(epoch_mean) 
        
        current_lr = optimizer.param_groups[0]['lr']
        if epoch_mean < best_val_l2:
            best_val_l2 = epoch_mean
            torch.save(model.state_dict(), best_model_path)
            status = "<--- save"
            patience_counter = 0
        else:
            status = f"[Wait {patience_counter+1}]"
            patience_counter += 1
            
        print(f"Epoch {epoch+1:03d} | LR: {current_lr:.2e} | Train Loss: {epoch_train_loss:.6f} | Val L2: {epoch_mean:.2f}% {status}")

        if patience_counter >= 30:
            print("=== EARLY STOPPING ===")
            break

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    return model

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
train_idx, val_idx, test_idx = get_indices(X_final.shape[0])

train_ds = SpatiotemporalDataset(X_final, train_idx)
val_ds = SpatiotemporalDataset(X_final, val_idx)
test_ds = SpatiotemporalDataset(X_final, test_idx)

BATCH_SIZE = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

MODES = (10, 24, 24)
HIDDEN_CHANNELS = 32
N_LAYERS = 4
EPOCHS = 150

best_model = train_model(
    modes=MODES, 
    hidden_channels=HIDDEN_CHANNELS, 
    n_layers=N_LAYERS,
    train_loader=train_loader, 
    val_loader=val_loader,
    static_mask=STATIC_MASK,
    device=device, 
    epochs=EPOCHS
)